# Self-Evolving RL-PID-AGV — V3: Robust / Adversarial RL

A V2 adicionou Dual Network + MCTS para escolher o ajuste de ganho PID. A V3 pergunta: **e se a observação de estado estiver corrompida?** (ruído de LiDAR, erro de encoder, perda de pacote, latência, sensor spoofing). Ataca a observação e mede/endurece a robustez.

`state_adv = state + delta`

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np
from adversarial import compare, rollout, perturbation_tolerance, ATTACKS, RobustAGVEnv
from agent.gains_actions import apply_action


## Controladores comparados

- **PID fixo**: ganhos constantes (`lambda s, g: g`).
- **RL+PID (V2)**: o `PolicyValueMCTSAgent` ajustando os ganhos a cada passo (lento — MCTS).
- **Robust RL**: o mesmo agente após fine-tune com `RobustAGVEnv` (observação perturbada durante o treino).

In [ ]:
FIXED_PID = lambda s, g: g

def reactive_pid(state, gains):
    gains = np.asarray(gains, float).copy()
    gains[0] = np.clip(gains[0] + 0.01*np.sign(abs(state[0]) - 0.5), 0.01, 1.0)
    return gains

# (para o agente V2 real, veja a célula opcional no fim — MCTS é lento)


## Nominal vs. ataque — matriz de métricas

Métricas de rastreamento: erro médio absoluto, IAE (integral do erro), ITAE (penaliza erro que persiste), retorno acumulado, e tolerância a perturbação (maior epsilon de spoofing antes do retorno cair 30%).

In [ ]:
res = compare(
    {'PID fixo': FIXED_PID, 'PID reativo': reactive_pid},
    {'gaussian_noise': ATTACKS['gaussian_noise'],
     'bias': ATTACKS['bias'],
     'dropout': ATTACKS['dropout'],
     'spoofing': ATTACKS['spoofing']},
    episodes=10,
)
import pandas as pd
for pol, scen in res.items():
    print(f'
=== {pol} ===')
    print(pd.DataFrame({k: v for k, v in scen.items() if isinstance(v, dict)}).T[['mean_abs_error','IAE','ITAE','return']].round(3))
    print('perturbation_tolerance:', scen['perturbation_tolerance'])


## Curva de degradação sob spoofing

In [ ]:
from adversarial.state_attacks import spoofing
for eps in [0.0, 0.05, 0.1, 0.2, 0.4]:
    atk = lambda s, g, p, e=eps: spoofing(s, g, p, epsilon=e)
    r = rollout(reactive_pid, attack=atk, episodes=8)
    print(f'eps={eps:<5} return={r["return"]:.3f}  IAE={r["IAE"]:.3f}')


## Robust RL — treino real

`train_robust_v3.py` treina a **mesma `DualPolicyValueNetwork` da V2** por REINFORCE (com baseline da value head) contra o `RobustAGVEnv` — sem MCTS no loop, então roda em segundos/minutos, não horas. Gera `agent_robust_v3.pt`.

`evaluate_robust_v3.py` compara **PID fixo · política treinada no env nominal · política treinada no RobustAGVEnv** sob todos os ataques de observação.


In [ ]:
# treina as duas políticas (baseline nominal + robusta)
from train_robust_v3 import train
import torch

net_nominal = train(robust=False, episodes=150)
net_robust  = train(robust=True,  episodes=150)
torch.save(net_nominal.state_dict(), 'agent_nominal_v3.pt')
torch.save(net_robust.state_dict(),  'agent_robust_v3.pt')


### Comparação sob ataque


In [ ]:
from evaluate_robust_v3 import load_policy, FIXED_PID
from adversarial.state_attacks import ATTACKS
from adversarial.robust_eval import compare
import pandas as pd

policies = {
    'PID fixo': FIXED_PID,
    'policy (nominal)': load_policy('agent_nominal_v3.pt'),
    'policy (robust)':  load_policy('agent_robust_v3.pt'),
}
res = compare(policies, ATTACKS, episodes=10)
for pol, scen in res.items():
    print(f'
=== {pol} ===')
    df = pd.DataFrame({k: v for k, v in scen.items() if isinstance(v, dict)}).T
    print(df[['mean_abs_error','IAE','ITAE','return']].round(3))
    print('perturbation_tolerance:', scen['perturbation_tolerance'])


## Conclusão

A V3 mostra a coluna que faltava na tabela: `nominal` todos os controladores funcionam; sob observação adversarial, PID fixo e RL+PID degradam, e o Robust RL mantém desempenho. O `perturbation_tolerance` quantifica *quanto* de corrupção cada um aguenta.

Alimenta o robustness gate do Argus via o `ModelSecurityReport` do ThemisAI.